In [56]:
import numpy as np
import pandas as pd

In [64]:
def best_gini(X, y):
    best_score, best_col, best_thresh = np.inf, None, None
    for col in range(X.shape[1]):
        s = np.sort(np.unique(X[:, col]))
        thresholds = [(s[i] + s[i+1])/2 for i in range(len(s) - 1)]

        for thresh in thresholds:
            score = split_score(X, y, col, thresh)
            if score < best_score:
                best_score, best_col, best_thresh = score, col, thresh
                
    return (best_col, best_score, best_thresh)
        
def split_score(X, y, feature_column, threshold):
    true_array = y[X[:, feature_column] >= threshold]
    false_array = y[X[:, feature_column] < threshold]

    n1, n2 = len(true_array), len(false_array)
    true_pct, false_pct = n1/(n1+n2), n2/(n1+n2)
    score = true_pct * gini(true_array) + false_pct * gini(false_array)

    return score

def gini(labels):
    _, freq = np.unique(labels, return_counts = True)
    n = len(labels)
    return 1- sum([(f/n)**2 for f in freq])
    

In [65]:
feature_names = ["smoker", "exercises", "owns_cat"]

X_toy = np.array([
    # smoker, exercises, owns_cat
    [1, 0, 1],   # sick
    [1, 0, 0],   # sick
    [1, 1, 1],   # sick
    [0, 0, 0],   # sick
    [1, 1, 0],   # healthy
    [0, 1, 1],   # healthy
    [0, 1, 0],   # healthy
    [0, 0, 1],   # healthy
], dtype=float)

y_toy = np.array([1, 1, 1, 1, 0, 0, 0, 0])   # 1 = sick

X_cont = np.array([1.0, 2.0, 3.0, 10.0]).reshape(-1, 1)   # shape (4, 1): 4 rows, 1 feature
y_cont = np.array([0,0,1,1])

print(best_gini(X_cont, y_cont))

(0, np.float64(0.0), np.float64(2.5))


In [60]:
from __future__ import annotations
from dataclasses import dataclass

@dataclass
class LeafNode:
    res: int

@dataclass
class DecisionNode:
    col: int
    threshold: float
    leftNode: DecisionNode | LeafNode | None = None
    rightNode: DecisionNode | LeafNode | None = None



In [ ]:


def build(X, y, depth = 2, score_cutoff = 0.01):   
    if not len(X) or not len(y):
        raise IndexError("Length was 0?")
    
    best_col, best_score, thresh = best_gini(X, y)
    curr = DecisionNode(best_col, thresh)

    if depth == 0 or gini(y) - best_score <= score_cutoff:
        majority = np.argmax(np.bincount(y))
        return LeafNode(majority)
    else:

        left_x, left_y = X[X[:,best_col] >= thresh], y[X[:,best_col] >= thresh]
        right_x, right_y = X[X[:,best_col] < thresh], y[X[:,best_col] < thresh]
        curr.leftNode = build(left_x, left_y, depth-1, score_cutoff)
        curr.rightNode = build(right_x, right_y, depth-1, score_cutoff)
    return curr

def predict(X, tree):
    ret = []
    for sample in X:
        curr = tree
        while curr and type(curr) is DecisionNode:
            c = curr.col
            if sample[c] >= curr.threshold:
                curr = curr.leftNode
            else:
                curr = curr.rightNode
        ret.append(curr.res if curr else None)
    return ret

In [71]:
from collections import deque
feature_names = ["smoker", "exercises", "owns_cat"]

def _display(curr: DecisionNode | LeafNode | None):
    if type(curr) is DecisionNode:
        print(f"col: {curr.col}, threshold: {curr.threshold:.2f}", end = " | ")
    elif type(curr) is LeafNode:
        print(f"res: {curr.res}", end = " | ")
    else:
        print(f"res : {None}")
        
def _display_graph(tree):
    q, newq = deque([tree]), deque([])
    while q or newq:
        while q:
            curr = q.pop()
            _display(curr)
            if not curr or type(curr) is not DecisionNode: continue
            for item in [curr.leftNode, curr.rightNode]:
                newq.appendleft(item)
        q, newq = newq, deque()
        print("", end ="\n")

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
X_toy = np.array([
    # smoker, exercises, owns_cat
    [1, 0, 1],   # sick
    [1, 0, 0],   # sick
    [1, 1, 1],   # sick
    [0, 0, 0],   # sick
    [1, 1, 0],   # healthy
    [0, 1, 1],   # healthy
    [0, 1, 0],   # healthy
    [0, 0, 1],   # healthy
], dtype=float)
y_toy = np.array([1, 1, 1, 1, 0, 0, 0, 0])   # 1 = sick

# clf = DecisionTreeClassifier(criterion= "gini", max_depth=2, random_state=2)
# clf.fit(X_toy, y_toy)
# sk_preds = clf.predict(X_toy)
# print(sk_preds)

node = build(X_toy, y_toy)

# print([int(res) for res in predict(X_toy, node)])

# print(export_text(clf, feature_names=feature_names))
# _display_graph(node)


In [78]:
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
X,y = make_classification(n_samples = 200, n_features = 4, random_state = 0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

clf = DecisionTreeClassifier(criterion= "gini", max_depth=6, random_state=2)
clf.fit(X_train, y_train)
node = build(X_train, y_train, depth = 6)

print(export_text(clf))
_display_graph(node)

sk_preds = clf.predict(X_test)
my_preds = predict(X_test, node)
print(sk_preds)
print([int(res) for res in my_preds])



|--- feature_0 <= 0.31
|   |--- feature_0 <= 0.05
|   |   |--- class: 0
|   |--- feature_0 >  0.05
|   |   |--- feature_0 <= 0.06
|   |   |   |--- feature_3 <= -0.06
|   |   |   |   |--- class: 0
|   |   |   |--- feature_3 >  -0.06
|   |   |   |   |--- class: 1
|   |   |--- feature_0 >  0.06
|   |   |   |--- class: 0
|--- feature_0 >  0.31
|   |--- feature_3 <= 0.11
|   |   |--- feature_0 <= 0.42
|   |   |   |--- feature_0 <= 0.38
|   |   |   |   |--- feature_2 <= -0.12
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- feature_2 >  -0.12
|   |   |   |   |   |--- feature_3 <= -0.37
|   |   |   |   |   |   |--- class: 1
|   |   |   |   |   |--- feature_3 >  -0.37
|   |   |   |   |   |   |--- class: 0
|   |   |   |--- feature_0 >  0.38
|   |   |   |   |--- class: 0
|   |   |--- feature_0 >  0.42
|   |   |   |--- class: 1
|   |--- feature_3 >  0.11
|   |   |--- class: 0

col: 0, threshold: 0.31 | 
col: 3, threshold: 0.11 | col: 0, threshold: 0.05 | 
res: 0 | col: 0, threshold: 0.42 | 

In [79]:
from sklearn.metrics import classification_report
# print(classification_report(y_test, sk_preds))
# print(classification_report(y_test, my_preds))
np.sum(np.array(my_preds) != sk_preds)

np.int64(0)

In [80]:
Xb, yb = load_breast_cancer(return_X_y=True)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(Xb, yb, test_size=0.2, random_state=0, stratify=yb)

clf = DecisionTreeClassifier(criterion= "gini", max_depth=6, random_state=2)
clf.fit(Xb_train, yb_train)
node = build(Xb_train, yb_train, depth = 6)

print(export_text(clf))
_display_graph(node)

sk_preds = clf.predict(Xb_test)
my_preds = predict(Xb_test, node)
print(sk_preds)
print([int(res) for res in my_preds])

|--- feature_27 <= 0.14
|   |--- feature_23 <= 952.90
|   |   |--- feature_13 <= 35.44
|   |   |   |--- feature_27 <= 0.13
|   |   |   |   |--- class: 1
|   |   |   |--- feature_27 >  0.13
|   |   |   |   |--- feature_1 <= 21.34
|   |   |   |   |   |--- feature_18 <= 0.01
|   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |--- feature_18 >  0.01
|   |   |   |   |   |   |--- class: 1
|   |   |   |   |--- feature_1 >  21.34
|   |   |   |   |   |--- class: 0
|   |   |--- feature_13 >  35.44
|   |   |   |--- feature_19 <= 0.00
|   |   |   |   |--- feature_21 <= 20.53
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- feature_21 >  20.53
|   |   |   |   |   |--- feature_14 <= 0.01
|   |   |   |   |   |   |--- class: 1
|   |   |   |   |   |--- feature_14 >  0.01
|   |   |   |   |   |   |--- class: 0
|   |   |   |--- feature_19 >  0.00
|   |   |   |   |--- class: 1
|   |--- feature_23 >  952.90
|   |   |--- feature_1 <= 18.50
|   |   |   |--- feature_11 <= 0.70
|   |   |   |   |---

In [83]:
print(classification_report(yb_test, sk_preds))
print(classification_report(yb_test, my_preds))
np.sum(np.array(my_preds) != sk_preds)

              precision    recall  f1-score   support

           0       0.89      0.93      0.91        42
           1       0.96      0.93      0.94        72

    accuracy                           0.93       114
   macro avg       0.92      0.93      0.93       114
weighted avg       0.93      0.93      0.93       114

              precision    recall  f1-score   support

           0       0.88      0.88      0.88        42
           1       0.93      0.93      0.93        72

    accuracy                           0.91       114
   macro avg       0.91      0.91      0.91       114
weighted avg       0.91      0.91      0.91       114



np.int64(2)